In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_probability as tfp
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
ROOT_DIR = r"----"  
IMG_SHAPE = (80, 95)
BATCH_SIZE = 4
EPOCHS = 25
TEST_SIZE = 0.15
VAL_SIZE = 0.15
SEED = 42

In [ ]:
def cargar_datos_pacientes(root_dir, etiquetas_csv):
    df_etiquetas = pd.read_csv(etiquetas_csv)
    pacientes_data = {}

    for idx, row in df_etiquetas.iterrows():
        paciente = row[0]
        alpha = row[-2]
        beta_val = row[-1]
        carpeta = os.path.join(root_dir, paciente)
        csv_path = os.path.join(carpeta, f"muestras_beta_{paciente}.csv")
        if not os.path.exists(csv_path):
            continue
        try:
            df = pd.read_csv(csv_path)
            imagenes = []
            for nombre in df["nombre_imagen"]:
                img_path = os.path.join(carpeta, nombre)
                if os.path.exists(img_path):
                    img = np.load(img_path)
                    if img.shape == IMG_SHAPE:
                        imagenes.append(img)
            if len(imagenes) == 400:
                imgs = np.stack(imagenes, axis=0)[..., np.newaxis]  # (400, 80, 95, 1)
                pacientes_data[paciente] = (imgs, np.array([alpha, beta_val], dtype=np.float32))
        except Exception as e:
            print(f"Error en paciente {paciente}: {e}")
    return pacientes_data

etiquetas_csv = os.path.join(ROOT_DIR, "etiquetas_filtradas_REALES.csv")
pacientes_data = cargar_datos_pacientes(ROOT_DIR, etiquetas_csv)


In [ ]:
pacientes = list(pacientes_data.keys())
train_pac, test_pac = train_test_split(pacientes, test_size=TEST_SIZE, random_state=SEED)
train_pac, val_pac = train_test_split(train_pac, test_size=VAL_SIZE / (1 - TEST_SIZE), random_state=SEED)

def construir_dataset(pacientes, pacientes_data, shuffle=False):
    X, y = [], []
    for p in pacientes:
        imgs, label = pacientes_data[p]
        X.append(imgs)
        y.append(label)

    X = np.stack(X, axis=0)
    y = np.stack(y, axis=0)

    ds = tf.data.Dataset.from_tensor_slices((X, y))

    if shuffle:
        ds = ds.shuffle(buffer_size=len(pacientes))

    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = construir_dataset(train_pac, pacientes_data, shuffle=True)
val_ds   = construir_dataset(val_pac,   pacientes_data, shuffle=False)
test_ds  = construir_dataset(test_pac,  pacientes_data, shuffle=False)




In [ ]:
def obtener_nombres_imagenes_por_paciente(pacientes, root_dir):
    nombres = []

    for p in pacientes:
        csv_imgs = os.path.join(root_dir, p, f"muestras_beta_{p}.csv")
        df = pd.read_csv(csv_imgs)

        for nombre in df["nombre_imagen"]:
            nombres.append(nombre)

    return nombres

In [ ]:
import tensorflow as tf

def build_model():
    
    inputs = tf.keras.Input(shape=(400, 80, 95, 1))

    x = tf.reduce_mean(inputs, axis=1)  

    x = tf.keras.layers.Rescaling(1./255)(x)

    x = tf.keras.layers.Conv2D(32, 5, activation='relu', padding='same')(x)
    x = tf.keras.layers.MaxPooling2D()(x)
    x = tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same')(x)
    x = tf.keras.layers.MaxPooling2D()(x)

    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(128, activation='softplus')(x) 
   
    raw_outputs = tf.keras.layers.Dense(2)(x)  
    outputs = tf.keras.layers.Lambda(lambda t: 1.0 + tf.nn.softplus(t))(raw_outputs)

    return tf.keras.Model(inputs, outputs)



In [ ]:
model = build_model()
model.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss='mse')
history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS)

In [ ]:

plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'][1:], label='Train Loss')
plt.plot(history.history['val_loss'][1:], label='Val Loss')
plt.title('Evolución de la pérdida (diferencia entre distribuciones beta)')
plt.xlabel('Épocas')
plt.ylabel('Error cuadrático medio')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:

import random


X_test, y_test = [], []
for batch_x, batch_y in test_ds:
    X_test.append(batch_x.numpy())
    y_test.append(batch_y.numpy())

X_test = np.concatenate(X_test, axis=0)
y_test = np.concatenate(y_test, axis=0)


with tf.device('/CPU:0'):
    
    y_pred = model.predict(X_test, batch_size=32) 
    alfa_pred = y_pred[:, 0]
    beta_pred = y_pred[:, 1]

   
    num_muestras = 5
    pacientes_a_mostrar = random.sample(range(len(X_test)), num_muestras)
    x_vals = np.linspace(0.01, 0.99, 200)

    plt.figure(figsize=(15, 10))
    for i, idx in enumerate(pacientes_a_mostrar, 1):
        real_dist = tfp.distributions.Beta(y_test[idx][0], y_test[idx][1])
        pred_dist = tfp.distributions.Beta(alfa_pred[idx], beta_pred[idx])

        
        real_vals = real_dist.prob(x_vals).numpy()
        pred_vals = pred_dist.prob(x_vals).numpy()

        plt.subplot(2, 3, i)
        plt.plot(x_vals, real_vals, label='Real', linewidth=2)
        plt.plot(x_vals, pred_vals, label='Predicha', linestyle='--')
        plt.title(
            f'Paciente {idx}\n'
            f'Real: α={y_test[idx][0]:.2f}, β={y_test[idx][1]:.2f}\n'
            f'Pred: α={alfa_pred[idx]:.2f}, β={beta_pred[idx]:.2f}'
        )
        plt.xlabel('x')
        plt.ylabel('Densidad')
        plt.legend()
       

    plt.tight_layout()
    plt.show()

In [ ]:


def calcular_moda_beta(alpha, beta):
    
    alpha = np.asarray(alpha)
    beta = np.asarray(beta)
    moda = np.where((alpha > 1) & (beta > 1),
                    (alpha - 1) / (alpha + beta - 2),
                    np.nan)  
    return moda


X_train, y_train = [], []
for batch_x, batch_y in train_ds:
    X_train.append(batch_x.numpy())
    y_train.append(batch_y.numpy())

X_train = np.concatenate(X_train, axis=0)
y_train = np.concatenate(y_train, axis=0)

with tf.device('/CPU:0'):
    
    y_pred_train = model.predict(X_train, batch_size=32)  
    alfa_pred_train = y_pred_train[:, 0]
    beta_pred_train = y_pred_train[:, 1]

    
    moda_real = calcular_moda_beta(y_train[:, 0], y_train[:, 1])
    moda_pred = calcular_moda_beta(alfa_pred_train, beta_pred_train)

   
    mask_valid = ~np.isnan(moda_real) & ~np.isnan(moda_pred)
    moda_real = moda_real[mask_valid]
    moda_pred = moda_pred[mask_valid]

    
    plt.figure(figsize=(7, 7))
    plt.scatter(moda_real, moda_pred, alpha=0.6, edgecolors='k')
    plt.plot([0, 1], [0, 1], 'r--', label='Línea ideal')
    plt.xlabel('Moda real')
    plt.ylabel('Moda predicha')
    plt.title('Comparación de modas en conjunto de entrenamiento')
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
num_total = len(moda_real)
num_invalid = np.isnan(moda_real).sum()
num_valid = num_total - num_invalid

print(f"Total de pacientes en entrenamiento: {num_total}")
print(f"Modas indefinidas (NaN): {num_invalid}")
print(f"Modas válidas: {num_valid}")

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(moda_real, moda_pred, alpha=0.6, edgecolors='k')
plt.plot([min(moda_real.min(), moda_pred.min()), max(moda_real.max(), moda_pred.max())],
         [min(moda_real.min(), moda_pred.min()), max(moda_real.max(), moda_pred.max())],
         'r--', label='Línea ideal')

plt.xscale('log')
plt.yscale('log')

plt.xlabel('Moda real (escala log)')
plt.ylabel('Moda predicha (escala log)')
plt.title('Comparación de modas (escala logarítmica)')

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:

plt.figure(figsize=(7, 7))
plt.scatter(moda_real, moda_pred, alpha=0.6, edgecolors='k')
plt.plot([1e-3, 1], [1e-3, 1], 'r--', label='Línea ideal')  
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Moda real (escala log)')
plt.ylabel('Moda predicha (escala log)')
plt.title('Comparación de modas (entrenamiento) en escala logarítmica')

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:

def obtener_alfa_beta(dataset, model, batch_size=32):
    X, y = [], []
    for batch_x, batch_y in dataset:
        X.append(batch_x.numpy())
        y.append(batch_y.numpy())
    X = np.concatenate(X, axis=0)
    y = np.concatenate(y, axis=0)

   
    with tf.device('/CPU:0'):
        y_pred = model.predict(X, batch_size=batch_size)

    alfa_pred = y_pred[:, 0]
    beta_pred = y_pred[:, 1]

    return X, y, alfa_pred, beta_pred


X_train, y_train, alfa_train, beta_train = obtener_alfa_beta(train_ds, model)
X_val, y_val, alfa_val, beta_val       = obtener_alfa_beta(val_ds, model)
X_test, y_test, alfa_test, beta_test   = obtener_alfa_beta(test_ds, model)

In [ ]:
import pandas as pd
import tensorflow as tf

def obtener_alfa_beta(dataset, model, nombre, batch_size=32):
    X, y = [], []
    for batch_x, batch_y in dataset:
        X.append(batch_x.numpy())
        y.append(batch_y.numpy())
    X = np.concatenate(X, axis=0)
    y = np.concatenate(y, axis=0)

   
    with tf.device('/CPU:0'):
        y_pred = model.predict(X, batch_size=batch_size)

    alfa_pred = y_pred[:, 0]
    beta_pred = y_pred[:, 1]

   
    df = pd.DataFrame({
        f"{nombre}_alfa": alfa_pred,
        f"{nombre}_beta": beta_pred
    })
    return df


df_train = obtener_alfa_beta(train_ds, model, "train")
df_val   = obtener_alfa_beta(val_ds, model, "val")
df_test  = obtener_alfa_beta(test_ds, model, "test")


df_total = pd.concat([df_train, df_val, df_test], axis=1)


df_total.to_excel("predicciones_alfa_beta_REALES.xlsx", index=False)

In [ ]:
import numpy as np
import tensorflow as tf


model2 = build_model()

def crear_dataset_con_pseudoetiquetas(dataset, modelo_previo, batch_size=32):
    X, y = [], []
    for batch_x, batch_y in dataset:
        X.append(batch_x.numpy())
        y.append(batch_y.numpy())
    X = np.concatenate(X, axis=0)

   
    with tf.device('/CPU:0'):
        y_pred = modelo_previo.predict(X, batch_size=batch_size)

 
    return X, y_pred


X_train, y_train_new = crear_dataset_con_pseudoetiquetas(train_ds, model)

from sklearn.model_selection import train_test_split
X_train_new, X_val_new, y_train_new, y_val_new = train_test_split(
    X_train, y_train_new, test_size=0.2, random_state=42
)

In [ ]:

model2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='MSE'    
)

In [ ]:
with tf.device('/CPU:0'):
    history = model2.fit(
        X_train_new, y_train_new,
        validation_data=(X_val_new, y_val_new),
        epochs=EPOCHS,
        batch_size=16
    )

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Val')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.title('Training and Validation Loss Curve')
plt.legend()
plt.show()

In [ ]:
log_error = np.log(moda_pred2) - np.log(moda_real2)

sigma_log = np.std(log_error, ddof=1)
print("Sigma empírica en log:", sigma_log)

x_line = np.logspace(
    np.log10(moda_real2.min()),
    np.log10(moda_real2.max()),
    300
)


y_ideal = x_line

y_plus1  = x_line * np.exp(sigma_log)
y_minus1 = x_line * np.exp(-sigma_log)


y_plus2  = x_line * np.exp(2*sigma_log)
y_minus2 = x_line * np.exp(-2*sigma_log)

import matplotlib.pyplot as plt

plt.figure(figsize=(7,7))

plt.scatter(
    moda_real2,
    moda_pred2,
    alpha=0.6,
    edgecolors='k',
    label='Train samples'
)

plt.plot(x_line, y_ideal, 'r--', label='y = x')


plt.plot(x_line, y_plus1,  'g-',  label=r'$\pm1\sigma$')
plt.plot(x_line, y_minus1,'g-')

plt.plot(x_line, y_plus2,  'b-',  label=r'$\pm2\sigma$')
plt.plot(x_line, y_minus2,'b-')

plt.xscale('log')
plt.yscale('log')

plt.xlabel('Modes from ground truth')
plt.ylabel('Predicted modes')
plt.title('Mode comparison in the training set')

plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
X_test, y_test_new = crear_dataset_con_pseudoetiquetas(test_ds, model)

In [ ]:
with tf.device('/CPU:0'):
    y_pred_test2 = model2.predict(X_test, batch_size=32)
    alfa_pred_test2 = y_pred_test2[:, 0]
    beta_pred_test2 = y_pred_test2[:, 1]

moda_real_test2 = calcular_moda_beta(y_test_new[:, 0], y_test_new[:, 1])
moda_pred_test2 = calcular_moda_beta(alfa_pred_test2, beta_pred_test2)

mask_valid_test2 = ~np.isnan(moda_real_test2) & ~np.isnan(moda_pred_test2)
moda_real_test2 = moda_real_test2[mask_valid_test2]
moda_pred_test2 = moda_pred_test2[mask_valid_test2]

plt.figure(figsize=(7, 7))
plt.scatter(moda_real_test2, moda_pred_test2, alpha=0.6, edgecolors='k')
plt.plot(
    [min(moda_real_test2.min(), moda_pred_test2.min()), max(moda_real_test2.max(), moda_pred_test2.max())],
    [min(moda_real_test2.min(), moda_pred_test2.min()), max(moda_real_test2.max(), moda_pred_test2.max())],
    'r--', label='Línea ideal'
)
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Moda real (escala log)')
plt.ylabel('Moda predicha (escala log)')
plt.title('Comparación de modas en conjunto de prueba (segundo modelo, escala logarítmica)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
with tf.device('/CPU:0'):
    y_pred_test2 = model2.predict(X_test, batch_size=32)
    alfa_pred_test2 = y_pred_test2[:, 0]
    beta_pred_test2 = y_pred_test2[:, 1]


moda_real_test2 = calcular_moda_beta(y_test_new[:, 0], y_test_new[:, 1])
moda_pred_test2 = calcular_moda_beta(alfa_pred_test2, beta_pred_test2)


mask_valid_test2 = ~np.isnan(moda_real_test2) & ~np.isnan(moda_pred_test2)
moda_real_test2 = moda_real_test2[mask_valid_test2]
moda_pred_test2 = moda_pred_test2[mask_valid_test2]


x_line = np.logspace(
    np.log10(moda_real_test2.min()),
    np.log10(moda_real_test2.max()),
    300
)

y_ideal = x_line


y_plus1  = x_line * np.exp(sigma_log)
y_minus1 = x_line * np.exp(-sigma_log)


y_plus2  = x_line * np.exp(2*sigma_log)
y_minus2 = x_line * np.exp(-2*sigma_log)


plt.figure(figsize=(7, 7))

plt.scatter(
    moda_real_test2,
    moda_pred_test2,
    alpha=0.6,
    edgecolors='k',
    label='Test samples'
)

plt.plot(x_line, y_ideal, 'r--', label='y = x')


plt.plot(x_line, y_plus1,  'g-', label=r'$\pm1\sigma$ (training)')
plt.plot(x_line, y_minus1,'g-')


plt.plot(x_line, y_plus2,  'b-', label=r'$\pm2\sigma$ (training)')
plt.plot(x_line, y_minus2,'b-')

plt.xscale('log')
plt.yscale('log')

plt.xlabel('Ground Truth mode (log scale)')
plt.ylabel('Predicted mode (log scale)')
plt.title('Mode comparison in the test set (empirical $\sigma$ after training)')

plt.legend()
plt.tight_layout()
plt.show()


In [ ]:

outside_2sigma = np.mean(np.abs(log_error) > 2*sigma_log)
outside_2sigma

In [ ]:
import numpy as np


log_error_test = np.log(moda_pred_test2) - np.log(moda_real_test2)


inside_1sigma = np.abs(log_error_test) <= sigma_log
inside_2sigma = np.abs(log_error_test) <= 2 * sigma_log


pct_1sigma = 100 * np.mean(inside_1sigma)
pct_2sigma = 100 * np.mean(inside_2sigma)

print(f"Porcentaje dentro de ±1σ: {pct_1sigma:.2f}%")
print(f"Porcentaje dentro de ±2σ: {pct_2sigma:.2f}%")


In [ ]:
from scipy.stats import beta
import numpy as np
import matplotlib.pyplot as plt


q_low = 0.025
q_high = 0.975


alfa_valid = alfa_pred_test2[mask_valid_test2]
beta_valid = beta_pred_test2[mask_valid_test2]
moda_pred_valid = moda_pred_test2


q_low_vals = beta.ppf(q_low, alfa_valid, beta_valid)
q_high_vals = beta.ppf(q_high, alfa_valid, beta_valid)


mask_q = (~np.isnan(q_low_vals)) & (~np.isnan(q_high_vals)) & (q_low_vals > 0)

q_low_vals = q_low_vals[mask_q]
q_high_vals = q_high_vals[mask_q]
moda_pred_valid = moda_pred_valid[mask_q]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta


gamma = 0.95
q = (1 - gamma) / 2


lower = beta.ppf(q, alfa_pred_test2, beta_pred_test2)
upper = beta.ppf(1 - q, alfa_pred_test2, beta_pred_test2)


moda_pred = (alfa_pred_test2 - 1) / (alfa_pred_test2 + beta_pred_test2 - 2)


lower = np.clip(lower, 1e-6, 1.0)
upper = np.clip(upper, 1e-6, 1.0)



In [ ]:
# Guardar en CSV
salida_csv = os.path.join(ruta_base, "etiquetas_test_REALES2.csv")
df_test.to_csv(salida_csv, index=False)

In [ ]:
csv_test_REALES = os.path.join(ruta_base, "etiquetas_test_REALES2.csv")
df_test.to_csv(csv_test_REALES, index=False)

print("Archivo guardado en:", csv_test_REALES)


In [ ]:
import pandas as pd
import os

ruta_base = r"C:\Users\malex\Documents\DOCTORADO\sexto_semestre\PLAN\PLAN2\WORK1"

csv_real = os.path.join(ruta_base, "etiquetas_test_REALES2.csv")

df_real = pd.read_csv(csv_real)
y_real = df_real['mode'].values.astype(float)

In [ ]:

print(len(y_real), len(moda_pred_test2))


In [ ]:
pacientes = list(pacientes_data.keys())

train_pac, test_pac = train_test_split(
    pacientes, test_size=TEST_SIZE, random_state=SEED
)

train_pac, val_pac = train_test_split(
    train_pac, test_size=VAL_SIZE / (1 - TEST_SIZE), random_state=SEED
)


In [ ]:
lower = moda_real_test2 * np.exp(-2 * sigma_log)
upper = moda_real_test2 * np.exp( 2 * sigma_log)


In [ ]:
inside = (moda_pred >= lower) & (moda_pred <= upper)
coverage = np.mean(inside)

print(f"Cobertura empírica ±2σ: {coverage*100:.2f}%")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import beta
from scipy.optimize import fsolve



var_add = 0.00075   


mask_valid = (~np.isnan(moda_pred)) & (moda_pred > 0) & (moda_pred < 1)
moda_pred = moda_pred[mask_valid]
y_real = y_real[mask_valid]

alpha_list = []
beta_list = []



for m in moda_pred:

    def system(params):
        a, b = params
        
       
        eq1 = (a - 1) / (a + b - 2) - m
        
        
        var = (a * b) / ((a + b)**2 * (a + b + 1))
        eq2 = var - var_add
        
        return [eq1, eq2]

    
    init_guess = [m * 50 + 2, (1 - m) * 50 + 2]

    sol = fsolve(system, init_guess, maxfev=5000)

    a_sol, b_sol = sol

    
    a_sol = max(a_sol, 1.0001)
    b_sol = max(b_sol, 1.0001)

    alpha_list.append(a_sol)
    beta_list.append(b_sol)

alpha_new = np.array(alpha_list)
beta_new = np.array(beta_list)


gamma = 0.95
q = (1 - gamma) / 2

lower = beta.ppf(q, alpha_new, beta_new)
upper = beta.ppf(1 - q, alpha_new, beta_new)


idx = np.argsort(y_real)
y_real_ord = y_real[idx]
moda_pred_ord = moda_pred[idx]
lower_ord = lower[idx]
upper_ord = upper[idx]
x = np.arange(len(y_real_ord))



x = np.arange(len(moda_pred))

plt.figure(figsize=(10, 6))

plt.vlines(
    x, lower, upper,
    color='gray',
    alpha=0.6,
    linewidth=2,
    label='Beta interval 95%'
)

plt.scatter(
    x, moda_pred,
    color='blue',
    s=35,
    zorder=3,
    label='Predicted mode'
)

plt.scatter(
    x, y_real,
    color='red',
    s=35,
    zorder=4,
    label='Ground truth mode'
)

plt.xlabel('Observations in Test Set')
plt.ylabel('PDFF')
plt.ylim(0, 0.2)
plt.title('Beta Prediction Intervals (95%)')
plt.legend()
plt.tight_layout()
plt.show()